# 02 Data Cleaning

Clean and transform the Olist datasets, create analysis fields and save processed data.


In [235]:
from pathlib import Path
import pandas as pd
import numpy as np


In [236]:
project_root = Path.cwd().parent

raw_data_path = project_root / "data" / "raw" / "Dataset"
processed_data_path = project_root / "data" / "processed"

processed_data_path.mkdir(parents = True, exist_ok = True)

print(raw_data_path)
print(processed_data_path)

/Users/xsha/Desktop/customer-product-analysis/data/raw/Dataset
/Users/xsha/Desktop/customer-product-analysis/data/processed


In [237]:
# Read all the csv files

customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")

orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")

order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")

products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")

reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")

category_translation = pd.read_csv(raw_data_path / "product_category_name_translation.csv")

In [238]:
# Check if the csv files can be read successfully

dataframes = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "reviews": reviews,
    "category_translation": category_translation
}

for name, df in dataframes.items():
    print(name, df.shape)

# (rows, columns)

customers (99441, 5)
orders (99441, 8)
order_items (112650, 7)
products (32951, 9)
reviews (99224, 7)
category_translation (71, 2)


In [239]:
# Use the copys of original datasets for data cleaning

customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
products_clean = products.copy()
reviews_clean = reviews.copy()
category_translation_clean = category_translation.copy()

In [240]:
# Clean the data formats

clean_dataframes = [
    customers_clean,
    orders_clean,
    order_items_clean,
    products_clean,
    reviews_clean,
    category_translation_clean
]

for df in clean_dataframes:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [241]:
# Check the duplicates

for name, df in {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "products": products_clean,
    "reviews": reviews_clean,
    "category_translation": category_translation_clean
}.items():
    print(name, df.duplicated().sum())

customers 0
orders 0
order_items 0
products 0
reviews 0
category_translation 0


In [242]:
# Delete the duplicates

customers_clean = customers_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
reviews_clean = reviews_clean.drop_duplicates()
category_translation_clean = category_translation_clean.drop_duplicates()

In [243]:
# Check if the major IDs in each sheets are unique

print("Unique customer_id:", customers_clean["customer_id"].is_unique)
print("Unique order_id:", orders_clean["order_id"].is_unique)
print("Unique product_id:", products_clean["product_id"].is_unique)


# Check if there are any duplicated items

duplicate_order_items = order_items_clean.duplicated(subset = ["order_id", "order_item_id"]).sum()
print("Duplicate order items:", duplicate_order_items)

Unique customer_id: True
Unique order_id: True
Unique product_id: True
Duplicate order items: 0


In [244]:
orders_clean.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [245]:
# Convert text to datetime

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for columns in order_date_columns:
    orders_clean[columns] = pd.to_datetime(orders_clean[columns], errors = 'coerce')


# Clean order_items

order_items_clean["shipping_limit_date"] = pd.to_datetime(order_items_clean["shipping_limit_date"], errors = 'coerce')


# Clean reviews

review_date_columns = ["review_creation_date", "review_answer_timestamp"]

for columns in review_date_columns:
    reviews_clean[columns] = pd.to_datetime(reviews_clean[columns], errors = 'coerce')

In [246]:
# Add purchase year and month in orders_clean

orders_clean["purchase_year"] = (orders_clean["order_purchase_timestamp"].dt.year)

orders_clean["purchase_month"] = (orders_clean["order_purchase_timestamp"].dt.month)

orders_clean["purchase_year_month"] = (orders_clean["order_purchase_timestamp"].dt.to_period("M").astype(str))


# Add purchase day of week and hour

orders_clean["purchase_day_of_week"] = (orders_clean["order_purchase_timestamp"].dt.day_name())

orders_clean["purchase_hour"] = (orders_clean["order_purchase_timestamp"].dt.hour)

In [247]:
# Calculate the actual delivery days

orders_clean["delivery_days"] = (orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]).dt.days


# Calculate the differences between acutal delivery days and estimated delivery days

orders_clean["delivery_delay_days"] = (orders_clean["order_delivered_customer_date"] - orders_clean["order_estimated_delivery_date"]).dt.days


# If it is delayed

orders_clean["is_delayed"] = np.where(orders_clean["delivery_delay_days"] > 0, 1, 0)


# For the missing delivery

missing_delivery_mash = (orders_clean["order_delivered_customer_date"].isna())

orders_clean.loc[missing_delivery_mash, "is_delayed"] = np.nan

In [248]:
# Check the results

orders_clean[[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delivery_delay_days",
        "is_delayed"
    ]].head()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,is_delayed
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,0.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,0.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,0.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,0.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,0.0


### Product category translation

In [249]:
category_translation_clean.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [250]:
# Merge

products_clean = products_clean.merge(category_translation_clean, on = "product_category_name", how = "left")

In [251]:
# Rename the column

products_clean = products_clean.rename(columns = {"product_category_name_english": "product_category"})

In [252]:
# Missing product categories

products_clean["product_category"] = (products_clean["product_category"].fillna("unknown"))

In [253]:
# Check the results

products_clean[[
    "product_id",
    "product_category_name",
    "product_category"
]].head()

,product_id,product_category_name,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


In [254]:
# Check the percentage of translations that were unsuccessful

unknown_category_count = (products_clean["product_category"] == "unknown").sum()

print("Unknown categories: ", unknown_category_count)

Unknown categories:  623


### Clean region formats

In [255]:
customers_clean["customer_city"] = (customers_clean["customer_city"].str.strip().str.lower())

customers_clean["customer_city"] = (customers_clean["customer_city"].str.strip().str.upper())

### Set up total price of products (price + delivery fees)

In [256]:
order_items_clean["item_total_value"] = (order_items_clean["price"] + order_items_clean["freight_value"])

In [257]:
# Check if there are any negative numbers

print("Negative prices:", (order_items_clean["price"] < 0).sum())

print("Negative freight:", (order_items_clean["freight_value"] < 0).sum())

Negative prices: 0
Negative freight: 0


In [258]:
order_items_clean[[
    "price",
    "freight_value",
    "item_total_value"
]].describe()

,price,freight_value,item_total_value
count,112650.000000,112650.000000,112650.000000
mean,120.653739,19.990320,140.644059
std,183.633928,15.806405,190.724394
min,0.850000,0.000000,6.080000
25%,39.900000,13.080000,55.220000
50%,74.990000,16.260000,92.320000
75%,134.900000,21.150000,157.937500
max,6735.000000,409.680000,6929.310000


In [259]:
# Check the highest price items

order_items_clean.nlargest(10, "price")[["order_id", "product_id", "price", "freight_value"]]

,order_id,product_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,3999.90,17.01


### Check the missing value in reviews

In [260]:
reviews_clean.isnull().sum()

# Some customers might just give a score but do not leave comment

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [261]:
reviews_clean["have_comment"] = np.where(reviews_clean["review_comment_message"].notna(), 1, 0)

In [262]:
reviews_clean["review_comment_message"] = (reviews_clean["review_comment_message"].fillna("").str.strip())

In [263]:
reviews_clean["review_score"].value_counts().sort_index()

# Scores should be between 1 to 5

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [264]:
invalid_socre = reviews_clean[~reviews_clean["review_score"].between(1, 5)]

print("Invlid review scores:", len(invalid_socre))

Invlid review scores: 0


### Check order status

In [265]:
orders_clean["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [266]:
orders_clean["is_complete"] = np.where(orders_clean["order_status"] == "deliveried", 1, 0)

### Final data quality check

In [267]:
cleaned_dataframes = {
    "customers_clean": customers_clean,
    "orders_clean": orders_clean,
    "order_items_clean": order_items_clean,
    "products_clean": products_clean,
    "reviews_clean": reviews_clean
}

for name, df in cleaned_dataframes.items():
    print(name, df.shape)

customers_clean (99441, 5)
orders_clean (99441, 17)
order_items_clean (112650, 8)
products_clean (32951, 10)
reviews_clean (99224, 8)


In [268]:
for name, df in cleaned_dataframes.items():
    print("\n", name)
    print(df.isnull().sum().sort_values(ascending=False).head(10))


 customers_clean
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

 orders_clean
is_delayed                       2965
delivery_delay_days              2965
order_delivered_customer_date    2965
delivery_days                    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
purchase_year_month                 0
purchase_hour                       0
purchase_day_of_week                0
dtype: int64

 order_items_clean
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
item_total_value       0
dtype: int64

 products_clean
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g         

In [269]:
for name, df in cleaned_dataframes.items():
    print("\n", name)
    print(df.dtypes)


 customers_clean
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

 orders_clean
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
purchase_year                             int32
purchase_month                            int32
purchase_year_month                      object
purchase_day_of_week                     object
purchase_hour                             int32
delivery_days                           float64
delivery_delay_days                     float64
is_delayed                              float64
is_complet

In [270]:
# Check if main IDs are missing

print(customers_clean["customer_id"].isna().sum())

print(orders_clean["order_id"].isna().sum())

print(order_items_clean["order_item_id"].isna().sum())

print(products_clean["product_id"].isna().sum())

0
0
0
0


### Save cleaned csv files

In [271]:
customers_clean.to_csv(processed_data_path / "customers_clean.csv", index = False)

orders_clean.to_csv(processed_data_path / "orders_clean.csv", index = False)

order_items_clean.to_csv(processed_data_path / "order_items_clean.csv", index = False)

products_clean.to_csv(processed_data_path / "products_clean.csv", index = False)

reviews.to_csv(processed_data_path / "reviews_clean.csv", index = False)

In [272]:
list(processed_data_path.glob("*.csv"))

[PosixPath('/Users/xsha/Desktop/customer-product-analysis/data/processed/reviews_clean.csv'),
 PosixPath('/Users/xsha/Desktop/customer-product-analysis/data/processed/reviews_mysql.csv'),
 PosixPath('/Users/xsha/Desktop/customer-product-analysis/data/processed/orders_clean.csv'),
 PosixPath('/Users/xsha/Desktop/customer-product-analysis/data/processed/customers_clean.csv'),
 PosixPath('/Users/xsha/Desktop/customer-product-analysis/data/processed/order_items_clean.csv'),
 PosixPath('/Users/xsha/Desktop/customer-product-analysis/data/processed/products_clean.csv')]

## Data Cleaning Summary

The following cleaning steps were completed:

1. Remove exact duplicate records.

2. Converted timestamp columns to datetime format.

3. Added purchase year, month, and year-month fields.

4. Calculated delivery time and delivery delay.

5. Created delayed delivery and completed order indicators.

6. Translated product categories into English.

7. Standardised customer city and state fields.

8. Create item total value using product price and freight value.

9. Retained missing review comments because written comments were optional.

10. Exported cleaned dataset for SQL analysis.

In [273]:
reviews_mysql = reviews_clean [[
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]].copy()

date_columns = ["review_creation_date", "review_answer_timestamp"]

for column in date_columns:
    reviews_mysql[column] = pd.to_datetime (reviews_mysql[column], errors = "coerce")

reviews_mysql.to_csv(
    "../data/processed/reviews_mysql.csv",
    index = False,
    encoding = "utf-8",
    date_format = "%Y-%m-%d %H:%M:%S"
)

reviews_mysql.head()

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01,2018-03-02 10:26:53
